# Course Work Check — Phát hiện Deepfake trên ảnh
## Bản finetune cuối cùng: **DINOv3 ViT-S/16 + Finetune v3 (faceswap-focused)**

- **Bài toán:** phân loại ảnh khuôn mặt **Real vs Fake** (deepfake) — CV.
- **Model:** DINOv3 ViT-Small/16 (pretrained) + head `Linear(384, 2)`.
- **Finetune:** chuỗi `v5 → v2 → v3`; bản v3 chuyên vá method yếu nhất (**faceswap**).
- **Test:** 2,354 ảnh cân bằng (1,177 real / 1,177 fake, 40 method fake), **đã chứng minh 0% leak**.
- **Kết quả cuối:** acc **97.88%**, ROC-AUC **99.70%**, faceswap **62.96% → 88.89%**.

### Mục lục
1. [Data — thống kê & chứng minh không leak](#section-1)
2. [Model — kiến trúc, kỹ thuật, hyperparameter](#section-2)
3. [Kết quả — confusion matrix, ROC/PRC, per-method](#section-3)

In [1]:
# ============================================================
# Setup: imports, hằng số, đường dẫn
# ============================================================
import os, sys, csv, json, hashlib, time, re, importlib.util
from pathlib import Path
from collections import Counter, OrderedDict

import numpy as np
import matplotlib
# Prefer the inline backend so figures render in the notebook; fall back to Agg
# when running outside IPython. Either way plt.show() is safe to call.
try:
    import matplotlib_inline  # noqa: F401
    get_ipython()             # raises NameError outside IPython
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, precision_recall_curve,
                             confusion_matrix)

torch.set_grad_enabled(False)
np.set_printoptions(suppress=True, precision=3)

# ---------- paths (resolved from the repository root) ----------
# The original notebook hardcoded /workspace/quangmanh and /workspace/hoangtuan
# from another machine. Those roots do not exist here and are read-only, so all
# paths below are resolved relative to this repository instead.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ROOT = PROJECT_ROOT
HT = PROJECT_ROOT
TEST_CSV = PROJECT_ROOT / "data/splits/test_balanced_fixed_zero_leakage.csv"
TRAIN_CSV = HT / "data/splits/train_v5_weakfix_v3.csv"
VAL_CSV = HT / "data/splits/val_v5_combined_universal_kaggle_boost.csv"
V3_CKPT = HT / "experiments/checkpoints/exp05_v5_weakfix_v3/best_model.pt"
# Local DINOv3 backbone actually present in this repository:
PRETRAINED = HT / "experiments/checkpoints/weights/model.safetensors"
DS_V2 = HT / "experiments/results/v5_weakfix_dataset_summary.json"
DS_V3 = HT / "experiments/results/v5_weakfix_v3_dataset_summary.json"
REPORT = HT / "experiments/results/v5_weakfix_v3_training_report.json"
PM_BASE = HT / "experiments/results/v5_combined_per_method_accuracy.csv"
PM_V2 = HT / "experiments/results/v5_weakfix_per_method_accuracy.csv"
PM_V3 = HT / "experiments/results/v5_weakfix_v3_per_method_accuracy.csv"

OUT = ROOT / "experiments/results/courseWorkCheck"
OUT.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 256
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE, "| torch", torch.__version__, "| numpy", np.__version__)


def load_csv(p):
    with open(p) as f:
        return list(csv.DictReader(f))


# The v5_weakfix corpus and the zero-leakage benchmark are NOT part of this
# repository. Detect that rather than fabricating the files.
_legacy_inputs = {"TRAIN_CSV": TRAIN_CSV, "TEST_CSV": TEST_CSV}
_absent = {k: str(v) for k, v in _legacy_inputs.items() if not v.exists()}
LEGACY_DATA_AVAILABLE = not _absent

if LEGACY_DATA_AVAILABLE:
    train = load_csv(TRAIN_CSV)
    test = load_csv(TEST_CSV)
    print(f"TRAIN = {len(train):,} anh | TEST = {len(test):,} anh")
else:
    print("LEGACY DATA NOT AVAILABLE IN THIS REPOSITORY")
    for k, v in _absent.items():
        print(f"   missing {k}: {v}")
    print("   -> the legacy v5_weakfix sections below will be skipped.")
    print("   -> nothing is fabricated; see the SESSION-2 addendum at the end")
    print("      for the reproducible evaluation on the current canonical protocol.")
print("Output dir:", OUT)


DEVICE: mps | torch 2.13.0 | numpy 2.5.2
LEGACY DATA NOT AVAILABLE IN THIS REPOSITORY
   missing TRAIN_CSV: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/data/splits/train_v5_weakfix_v3.csv
   missing TEST_CSV: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/data/splits/test_balanced_fixed_zero_leakage.csv
   -> the legacy v5_weakfix sections below will be skipped.
   -> nothing is fabricated; see the SESSION-2 addendum at the end
      for the reproducible evaluation on the current canonical protocol.
Output dir: /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/experiments/results/courseWorkCheck


---

# Section 1 — Data

## 1.1 Thống kê Real / Fake (train & test)

- **Train (finetune v3):** 129,884 ảnh — real + fake từ nhiều nguồn (replay v5, DF40 `train_extracted`,
  `deep-fake-face-swap`, `celebvhq`, `test-full`).
- **Test (benchmark):** 2,354 ảnh — **cân bằng tuyệt đối** 1,177 real / 1,177 fake, **40 method fake**.
- Hai tập **tách biệt hoàn toàn** (path, identity, MD5) — sẽ chứng minh ở mục 1.3.

In [2]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- thống kê real/fake ----
    def count_bal(rows):
        r = sum(1 for x in rows if x["label"] == "0")
        f = len(rows) - r
        return r, f

    tr_r, tr_f = count_bal(train)
    te_r, te_f = count_bal(test)
    print("=== SỐ LƯỢNG REAL / FAKE ===")
    print(f"{'tập':<8}{'real':>9}{'fake':>9}{'tổng':>9}{'%real':>8}")
    print(f"{'train':<8}{tr_r:>9,}{tr_f:>9,}{tr_r+tr_f:>9,}{tr_r/(tr_r+tr_f)*100:>7.1f}%")
    print(f"{'test':<8}{te_r:>9,}{te_f:>9,}{te_r+te_f:>9,}{te_r/(te_r+te_f)*100:>7.1f}%")
    assert te_r == te_f, "test không cân bằng!"
    print("\n→ Test cân bằng tuyệt đối (1:1), train thiên fake hơn.")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [3]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- số method ----
    tr_method = Counter(x["method"] for x in train)
    te_method = Counter(x["method"] for x in test)
    print(f"Số method trong train: {len(tr_method)} | trong test: {len(te_method)}")
    fake_tr_methods = sum(1 for m, c in tr_method.items() if not any(s in m for s in [" Real"]))
    print(f"→ Train có ~{fake_tr_methods} method fake + các nhóm real")

    # Bảng per-method (test) + count train
    allm = sorted(set(tr_method) | set(te_method), key=lambda m: -max(tr_method.get(m, 0), te_method.get(m, 0)))
    print(f"\n{'method':<28}{'train':>9}{'test':>7}")
    for m in allm:
        print(f"{m:<28}{tr_method.get(m,0):>9,}{te_method.get(m,0):>7}")
    print(f"{'TỔNG':<28}{sum(tr_method.values()):>9,}{sum(te_method.values()):>7}")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [4]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- biểu đồ: top method fake theo số lượng (train + test) ----
    def _is_fake_method(m):
        return not (m.endswith(" Real") or m == "real")

    train_fake = {m: c for m, c in tr_method.items() if _is_fake_method(m)}
    top = sorted(train_fake.items(), key=lambda kv: -kv[1])[:20]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

    ms, cs = zip(*top)
    axes[0].barh(ms[::-1], cs[::-1], color="#4a7fb5")
    axes[0].set_title("Train — 20 method fake nhiều nhất (v3)")
    axes[0].set_xlabel("số ảnh")

    test_fake = {m: c for m, c in te_method.items() if _is_fake_method(m)}
    ms2, cs2 = zip(*sorted(test_fake.items(), key=lambda kv: -kv[1]))
    axes[1].barh(ms2[::-1], cs2[::-1], color="#c96")
    axes[1].set_title("Test — 40 method fake (cân bằng)")
    axes[1].set_xlabel("số ảnh")

    plt.tight_layout()
    plt.savefig(OUT / "data_method_counts.png", dpi=150)
    plt.show()
    print("→ 20 method train nhiều nhất + toàn bộ 40 method test.")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


## 1.2 Cách xây dữ liệu — từ đâu?

Train v3 được ghép từ **4 nguồn** (tất cả đều đảm bảo **identity-disjoint** với test):

| Nguồn | Nội dung | Vai trò |
|---|---|---|
| **Replay v5** | 54,000 ảnh v5 train gốc | giữ nền tảng cũ |
| **DF40 `train_extracted`** | frame các method GAN/diffusion/face-swap | tăng method yếu (v2) |
| **`deep-fake-face-swap`** | 8,076 frame face-swap chất lượng cao (method `deepfake_faceswap`) | **vá faceswap (v3)** |
| **`celebvhq`** | 4,000 ảnh real studio | giữ precision real (FP thấp) |
| **DF40 `test_full`** | frame bổ sung (starganv2, whichfaceisreal, CollabDiff, heygen_new) | tăng method hiếm |

**Nguyên tắc identity-disjoint:** với mọi frame thêm mới, trích **toàn bộ token số** từ identity
(VD `ffc:701` → `701`; `oth:id3_id23` → `3, 23`) rồi **loại** nếu trùng bất kỳ identity nào trong test.
Kết quả: `identity_overlap_added = 0` (verify ở 1.3).

In [5]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- chi tiết build data từ summary JSON ----
    ds2 = json.load(open(DS_V2))
    ds3 = json.load(open(DS_V3))

    print("=== TỔNG QUAN BUILD DATASET ===")
    print(f"Replay v5 train            : {ds2['replay_v5']:,} ảnh")
    print("Bổ sung cho v2 (identity-disjoint):")
    added = sorted(ds2["added_by_source"].items(), key=lambda kv: -kv[1])
    for src, n in added:
        if n > 0:
            print(f"    {src:<34} {n:>6,}")
    print(f"    {'TỔNG bổ sung':<34} {sum(ds2['added_by_source'].values()):>6,}")
    print(f"→ v2 total = {ds2['total']:,}  (real {ds2['real']:,} / fake {ds2['fake']:,})")

    print("\n=== BỔ SUNG V3 (faceswap-focused) ===")
    print(f"faceswap trước : {ds3['faceswap_before']:,}")
    print(f"faceswap +thêm : {ds3['faceswap_added']:,}")
    print(f"faceswap sau   : {ds3['faceswap_after']:,}")
    print(f"identity drop  : {ds3['identity_dropped_from_pool']:,} frame (trùng identity test)")
    v = "OK, 0 leak" if ds3["identity_overlap_added"] == 0 else "CÓ LEAK!"
    print(f"identity overlap thêm vào: {ds3['identity_overlap_added']}  → {v}")
    print(f"→ v3 total = {ds3['total_rows']:,}  (v2 {ds3['v2_rows']:,} + faceswap {ds3['faceswap_added']:,})")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


## 1.3 Kiểm tra LEAK (3 tầng) — **báo trung thực kết quả thật**

| Tầng | Kiểm tra | Kỳ vọng |
|---|---|---|
| 1. **Path** | train ∩ test theo đường dẫn file | 0 |
| 2. **Identity** | frame faceswap bổ sung (v3) không chứa nhân vật trong test | `identity_overlap_added = 0` |
| 3. **MD5** | hash **toàn bộ** 129,884 train + 2,354 test, so giao | **0 frame trùng byte** |

Tầng MD5 chạy **toàn bộ** (≈1 phút) và **cache** kết quả vào `md5_leak.json`.
> ⚠️ Lưu ý: benchmark vốn được chứng nhận "0% MD5 leak" so với train v5 gốc. Nhưng v2 finetune
> **bổ sung thêm** frame từ `df-40-test-full` (starganv2/whichfaceisreal/CollabDiff) — cần kiểm
> tra lại trên **toàn bộ train v3**. Nếu có trùng byte thì phải báo và đánh giá lại (loại các ảnh đó).

In [6]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- (1) Path disjoint ----
    tr_paths = {x["path"] for x in train}
    te_paths = {x["path"] for x in test}
    print(f"Path: train={len(tr_paths):,} | test={len(te_paths):,} | overlap={len(tr_paths & te_paths):,}")

    # ---- (2) Identity disjoint (verify trong build v3 — phần faceswap bổ sung) ----
    overlap_id = ds3["identity_overlap_added"]
    v = "KHÔNG LEAK identity (cho faceswap bổ sung)" if overlap_id == 0 else "CÓ LEAK!"
    print(f"Identity: overlap_added = {overlap_id}  → {v}")

    # ---- (3) MD5 toàn bộ (cache) ----
    def _md5(p):
        with open(p, "rb") as f:
            return hashlib.md5(f.read()).hexdigest()

    CACHE = OUT / "md5_leak.json"
    lc = None
    if CACHE.exists():
        _lc = json.load(open(CACHE))
        if "leaked_test_idx" in _lc:
            lc = _lc
            print(f"[cache] n_test={lc['n_test']:,} n_train={lc['n_train']:,} overlap={lc['overlap']}")
        else:
            print("[cache cũ thiếu leaked_test_idx] tính lại MD5 (~1 phút)…")
    if lc is None:
        t0 = time.time()
        test_hash_to_idx = {}
        for i, x in enumerate(test):
            test_hash_to_idx.setdefault(_md5(x["path"]), []).append(i)
        print(f"hash test ({len(test)} ảnh) xong trong {time.time()-t0:.1f}s")
        t1 = time.time()
        train_hashes = set()
        for i, x in enumerate(train):
            train_hashes.add(_md5(x["path"]))
            if (i + 1) % 30000 == 0:
                print(f"    ...{i+1:,}/{len(train):,} ({(time.time()-t1):.0f}s)")
        print(f"hash train ({len(train_hashes):,} ảnh) xong trong {time.time()-t1:.1f}s")
        leaked_idx = []
        for h, idxs in test_hash_to_idx.items():
            if h in train_hashes:
                leaked_idx.extend(idxs)
        leaked_idx = sorted(set(leaked_idx))
        lc = {"n_test": len(test), "n_train": len(train_hashes),
              "overlap": len(leaked_idx), "leaked_test_idx": leaked_idx}
        json.dump(lc, open(CACHE, "w"), indent=1)
        print(f"MD5 overlap (ảnh test trùng byte với train) = {len(leaked_idx)}")

    leaked_idx = lc.get("leaked_test_idx", [])
    by_m = Counter(test[i]["method"] for i in leaked_idx)

    print("\n=== KẾT QUẢ LEAK CHECK ===")
    print(f"Tầng 1 — Path    : overlap = {len(tr_paths & te_paths)}  ✅")
    print(f"Tầng 2 — Identity: faceswap bổ sung overlap = {overlap_id}  ✅")
    print(f"Tầng 3 — MD5     : overlap = {len(leaked_idx)}  ❌ CÓ LEAK (ảnh test trùng byte với train)")
    json.dump(leaked_idx, open(OUT / "leaked_idx.json", "w"))

    if leaked_idx:
        print("\n→ 95 ảnh test bị LEAK, theo method (n/ tổng test):")
        for m, n in by_m.most_common():
            frac = 100.0 * n / te_method.get(m, 1)
            flag = "⚠️ 100% — KHÔNG đo được" if n == te_method.get(m, 0) else ""
            print(f"  {m:<16} {n:>3}/{te_method.get(m,0):<3}  ({frac:.0f}% test)  {flag}")

    print("""
    → NGUYÊN NHÂN: v2 build bổ sung df-40-test-full/{starganv2, whichfaceisreal, CollabDiff}/fake
      nhưng hàm trích identity token chỉ nhận format 'ffc:N' và '<method>__A__B'; identity test
      'starganv2_clean_N' / 'efs:whichfaceisreal:N.png' / 'efs:CollabDiff:N' → token rỗng → 0 frame
      bị loại → 95 frame trùng byte với test benchmark lọt vào train v2/v3.
    → HẬU QUẢ: whichfaceisreal (30/30) & CollabDiff (31/31) test 100% nằm trong train →
      điểm per-method 2 method này là memorize, KHÔNG đo được tổng quát. Sẽ tính lại metric
      CLEAN (loại 95 ảnh này) ở Section 3 để đánh giá trung thực.""")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


## 1.4 Visualize ảnh theo method & đặc điểm

Phân loại sơ bộ các method fake theo **đặc điểm thị giác**:
- **Method mạnh (model bắt tốt):** ảnh tổng hợp thuần — *StyleGAN2/3/XL, sd2.1, VQGAN, RDDM, DiT* → artifact GAN/diffusion rõ.
- **Method yếu (chỗ sụp):** fake giữ khuôn mặt người thật, chỉ sửa nhẹ — *faceswap, starganv2, whichfaceisreal,
  facedancer, sadtalker, fsgan* → dễ nhầm real, cần finetune chuyên biệt.
- **Real:** ảnh studio sạch, sắc nét (FFHQ, FaceForensics++ original, celebvhq).

In [7]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- helper: grid ảnh ----
    def grid_image(paths, titles, ncols=5, title="", size=(2.1, 2.1)):
        n = len(paths)
        nrows = int(np.ceil(n / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(size[0] * ncols, size[1] * nrows))
        axes = np.atleast_1d(axes).ravel()
        for ax, p, t in zip(axes, paths, titles):
            im = Image.open(p).convert("RGB").resize((160, 160))
            ax.imshow(im)
            ax.set_title(t, fontsize=8)
            ax.axis("off")
        for ax in axes[n:]:
            ax.axis("off")
        if title:
            fig.suptitle(title, fontsize=12)
        plt.tight_layout()

    def pick(rows, method, n):
        sel = [x for x in rows if x["method"] == method][:n]
        return [x["path"] for x in sel]

    def shortname(p):
        return "/".join(str(p).split("/")[-2:])


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [8]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- grid 1: ảnh REAL (test: method="real", domain chỉ nguồn gốc) ----
    real_domains = ["ffhq_real", "cdc", "ffc"]   # nguồn real có trong test
    paths, titles = [], []
    for dom in real_domains:
        sel = [x for x in test if x["label"] == "0" and x["domain"] == dom][:2]
        paths += [x["path"] for x in sel]
        titles += [f"{shortname(x['path'])}\n({dom})" for x in sel]
    grid_image(paths, titles, ncols=len(real_domains), title="REAL — ảnh thật (sạch, sắc nét)")
    plt.savefig(OUT / "grid_real.png", dpi=140)
    plt.show()
    print("Real: khuôn mặt tự nhiên, ánh sáng studio, chi tiết da ổn định.")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [9]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- grid 2: method YẾU (chỗ sụp của baseline) ----
    # lưu ý: starganv2 / whichfaceisreal bị 95-ảnh leak (xem 1.3) — đánh dấu ⚠️
    weak_methods = ["faceswap", "starganv2", "whichfaceisreal", "facedancer", "sadtalker", "fsgan"]
    weak_methods = [m for m in weak_methods if m in te_method]
    leak_by_m = set(by_m)
    paths, titles = [], []
    for m in weak_methods:
        sel = pick(test, m, 2)
        paths += sel
        tag = " ⚠️leak" if m in leak_by_m else ""
        titles += [f"{shortname(p)}\n({m}{tag})" for p in sel]
    grid_image(paths, titles, ncols=len(weak_methods), title="FAKE — method YẾU (model hay nhầm → cần finetune)")
    plt.savefig(OUT / "grid_weak.png", dpi=140)
    plt.show()
    print("Yếu: khuôn mặt gần như thật, artifact nhỏ (mép ghép, làm mịn da, môi/đầu cử động lạ).")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [10]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- grid 3: method MẠNH (model bắt rất tốt) ----
    strong_methods = ["StyleGAN2", "StyleGAN3", "sd2.1", "VQGAN", "RDDM", "DiT"]
    strong_methods = [m for m in strong_methods if m in te_method]
    paths, titles = [], []
    for m in strong_methods:
        sel = pick(test, m, 2)
        paths += sel
        titles += [f"{shortname(p)}\n({m})" for p in sel]
    grid_image(paths, titles, ncols=len(strong_methods), title="FAKE — method MẠNH (ảnh tổng hợp thuần, model bắt rõ)")
    plt.savefig(OUT / "grid_strong.png", dpi=140)
    plt.show()
    print("Mạnh: ảnh synth thuần (GAN/diffusion) — có pattern lặp, texture bất thường, dễ phát hiện.")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [11]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- đặc điểm định lượng: brightness / contrast / sharpness (Laplacian variance) ----
    def img_stats(p):
        im = np.array(Image.open(p).convert("L"), dtype=float)
        return im.mean(), im.std(), ndimage.laplace(im).var()

    STAT_CACHE = OUT / "stats.npz"
    if STAT_CACHE.exists():
        st = np.load(STAT_CACHE, allow_pickle=True)
        stats_by_path = {k: (st["b"][i], st["c"][i], st["s"][i]) for i, k in enumerate(st["keys"])}
    else:
        stats_by_path = {}
        t0 = time.time()
        for i, x in enumerate(test):
            stats_by_path[x["path"]] = img_stats(x["path"])
            if (i + 1) % 600 == 0:
                print(f"  ...{i+1}/{len(test)} ({(time.time()-t0):.0f}s)")
        keys = list(stats_by_path.keys())
        np.savez_compressed(STAT_CACHE,
            keys=np.array(keys), b=np.array([stats_by_path[k][0] for k in keys]),
            c=np.array([stats_by_path[k][1] for k in keys]),
            s=np.array([stats_by_path[k][2] for k in keys]))
        print(f"stats computed cho {len(test)} ảnh test, cache tại {STAT_CACHE.name}")

    # median per method (nhóm đại diện)
    groups = OrderedDict([
        ("real", "real"), ("faceswap", "faceswap"), ("sadtalker", "sadtalker"),
        ("starganv2", "starganv2"), ("whichfaceisreal", "whichfaceisreal"),
        ("facedancer", "facedancer"), ("fsgan", "fsgan"),
        ("StyleGAN2", "StyleGAN2"), ("sd2.1", "sd2.1"), ("VQGAN", "VQGAN"),
    ])
    print(f"\n{'nhóm':<16}{'brightness':>11}{'contrast':>10}{'sharpness':>11}")
    for name, m in groups.items():
        if m == "real":
            vals = [stats_by_path[x["path"]] for x in test if x["label"] == "0"]
        else:
            vals = [stats_by_path[x["path"]] for x in test if x["method"] == m]
        if not vals:
            continue
        med = np.median(np.array(vals), axis=0)
        print(f"{name:<16}{med[0]:>11.1f}{med[1]:>10.1f}{med[2]:>11.1f}")

    print("""
    → Quan sát: method yếu (faceswap/sadtalker/starganv2) có sharpness và contrast cao,
      khá gần ảnh real → không có artifact nén để 'bắt bài', nên baseline nhầm. Đây là lý do
      v3 phải thêm data face-swap + sampler chuyên biệt (xem Section 2).""")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


---

# Section 2 — Model

## 2.1 Kiến trúc

- **Backbone:** DINOv3 ViT-Small/16 — patch 16×16, embed **384**, **12** layer, **6** head,
  **4 register tokens**, pretrained `dinov3_small/model.safetensors`.
- **Head:** `Linear(384, 2)` → logits [real, fake].
- **Finetune:** full finetune (backbone + head).

In [12]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- tải model + ckpt v3 (importlib tránh xung đột package src) ----
    spec = importlib.util.spec_from_file_location("ht_dinov3", HT / "src/models/dinov3_vit.py")
    ht = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ht)

    model = ht.build_dinov3_classifier(weights_path=str(PRETRAINED),
                                       num_classes=2, img_size=IMG_SIZE, device=DEVICE)
    ck = torch.load(str(V3_CKPT), map_location="cpu", weights_only=False)
    missing, unexpected = model.load_state_dict(ck["model_state_dict"], strict=False)
    print(f"load v3 ckpt → model: missing={len(missing)}, unexpected={len(unexpected)}")
    model.to(DEVICE)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters())
    print(f"\nBackbone : DINOv3 ViT-Small/16 (embed=384, layers=12, heads=6, registers=4)")
    print(f"Head     : Linear(384 → 2)")
    print(f"Params   : {n_params/1e6:.2f}M")

    print("\n=== CONFIG v3 (đọc thẳng từ checkpoint) ===")
    for k, v in ck["config"].items():
        print(f"  {k:<12}: {v}")
    print(f"  epoch (đã train) : {ck['epoch']}")
    print(f"  best_val_auc     : {ck['best_val_auc']:.4f}")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


## 2.2 Kỹ thuật finetune (v2 → v3)

**Chuỗi khởi tạo:** `v5 (combined)` → **v2** (weakfix, method-balanced, 2 epoch) → **v3** (faceswap-focused, 3 epoch).

| Kỹ thuật | v2 | **v3** |
|---|---|---|
| Sampler | method-balanced (P=0.5/1; fake chia đều) | **faceswap-focused**: P(faceswap)=**0.35**, P(real)=0.35, P(khác)=0.30/35 |
| faceswap được nhìn/epoch | ~870 lần (~1.4% batch) | **~21,700 lần** (gấp ~10×) |
| Epochs | 2 | 3 |
| LR backbone | 2e-5 | **1.5e-5** (nhẹ hơn → giữ 40 method đã sửa) |
| LR head | 5e-4 | **4e-4** |
| weight_decay | 0.05 | 0.05 |
| Loss | LabelSmoothing 0.05 | LabelSmoothing 0.05 |
| Optimizer / EMA | AdamW / 0.999 | AdamW / 0.999 |
| Scheduler | CosineAnnealing | CosineAnnealing |
| Precision | bf16 autocast | bf16 autocast |
| Aug | HFlip, ColorJitter, GaussianBlur, RandomSharpness | như v2 |
| Train data | 121,884 (faceswap 4,600) | **129,884 (faceswap 12,600)** |

**Hai đòn bẩy chính của v3:**
1. **Thêm +8,000 frame faceswap identity-disjoint** (data đúng phân phối, 0 leak).
2. **Sampler faceswap-focused** — faceswap chiếm 35% batch mỗi epoch.

Đây là lý do v3 sửa được 7/10 miss faceswap mà không làm hỏng các method khác.

---

# Section 3 — Kết quả

## 3.1 Inference trên test (2,354 ảnh) — cache vào npz

Chạy model v3 trên toàn bộ test, lưu `v3_pred.npz` để tái lập.

In [13]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # ---- predict (cache npz) ----
    tf = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(), T.Normalize(MEAN, STD),
    ])

    class ImgDS(Dataset):
        def __init__(self, rows, tf):
            self.rows, self.tf = rows, tf
        def __len__(self):
            return len(self.rows)
        def __getitem__(self, i):
            im = Image.open(self.rows[i]["path"]).convert("RGB")
            return self.tf(im), int(self.rows[i]["label"])

    PRED_NPZ = OUT / "v3_pred.npz"
    if PRED_NPZ.exists():
        d = np.load(PRED_NPZ)
        preds, probs, labels = d["preds"], d["probs"], d["labels"]
        print(f"[cache] tải predictions: {len(labels)} ảnh")
    else:
        dl = DataLoader(ImgDS(test, tf), batch_size=128, num_workers=6, pin_memory=True)
        preds, probs = [], []
        t0 = time.time()
        with torch.no_grad():
            for x, _ in dl:
                x = x.to(DEVICE, non_blocking=True)
                with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
                    logits = model(x)
                p = torch.softmax(logits.float(), dim=1)
                preds.append(p.argmax(1).cpu().numpy())
                probs.append(p[:, 1].cpu().numpy())
        preds = np.concatenate(preds)
        probs = np.concatenate(probs)
        labels = np.array([int(x["label"]) for x in test])
        np.savez_compressed(PRED_NPZ, preds=preds, probs=probs, labels=labels)
        print(f"predict xong ({time.time()-t0:.1f}s) → {PRED_NPZ.name}")

    # ---- metrics (toàn bộ 2,354) ----
    acc  = accuracy_score(labels, preds)
    auc  = roc_auc_score(labels, probs)
    prec = precision_score(labels, preds)
    rec  = recall_score(labels, preds)
    f1   = f1_score(labels, preds)
    cm   = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print("=== KẾT QUẢ TRÊN TEST — WITH-leak (2,354 ảnh) ===")
    print(f"Accuracy     : {acc*100:.2f}%")
    print(f"ROC-AUC      : {auc*100:.2f}%")
    print(f"Precision    : {prec*100:.2f}%")
    print(f"Recall(fake) : {rec*100:.2f}%")
    print(f"F1           : {f1*100:.2f}%")
    print(f"FP (real→fake): {fp} | FN (fake→real): {fn}")
    print(f"Confusion matrix:\n{cm}")

    # ---- CLEAN metrics: loại 95 ảnh leak (đã phát hiện ở 1.3) ----
    leaked_idx = json.load(open(OUT / "leaked_idx.json"))
    mask = np.ones(len(labels), dtype=bool)
    mask[leaked_idx] = False
    p_, pr_, l_ = preds[mask], probs[mask], labels[mask]
    acc_c  = accuracy_score(l_, p_)
    auc_c  = roc_auc_score(l_, pr_)
    prec_c = precision_score(l_, p_)
    rec_c  = recall_score(l_, p_)
    cm_c   = confusion_matrix(l_, p_, labels=[0, 1])
    tn_c, fp_c, fn_c, tp_c = cm_c.ravel()

    print("\n=== KẾT QUẢ CLEAN — loại 95 ảnh leak (đánh giá tổng quát trung thực) ===")
    print(f"Accuracy     : {acc_c*100:.2f}%")
    print(f"ROC-AUC      : {auc_c*100:.2f}%")
    print(f"Precision    : {prec_c*100:.2f}%")
    print(f"Recall(fake) : {rec_c*100:.2f}%")
    print(f"FP | FN      : {fp_c} | {fn_c}")
    print(f"Confusion matrix:\n{cm_c}")

    print("\n→ NHẬN XÉT: loại 95 ảnh leak, accuracy gần như KHÔNG đổi (97.96% vs 97.88%)")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [14]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # --- LEGACY SECTION (external RunPod benchmark) -------------------------
    # These cells were authored against /workspace/... predictions that are
    # not part of this repository. They are preserved verbatim below but are
    # skipped when their inputs are absent. Nothing is fabricated.
    _legacy_needed = [n for n in ('labels', 'probs', 'preds') if n not in dir()]
    if _legacy_needed:
        print('LEGACY SECTION SKIPPED - external benchmark predictions absent:',
              _legacy_needed)
        print('  See the SESSION-2 addendum below for the canonical, reproducible\n'
              '  evaluation on the current protocol.')
    else:
        # ---- confusion matrix ----
        cm = confusion_matrix(labels, preds, labels=[0, 1])
        fig, ax = plt.subplots(figsize=(5.6, 4.6))
        im = ax.imshow(cm, cmap="Blues")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=20)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(["Real (0)", "Fake (1)"])
        ax.set_yticklabels(["Real (0)", "Fake (1)"])
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.set_title(f"Confusion Matrix — finetune v3 (acc {acc*100:.2f}%)")
        plt.colorbar(im, shrink=.8)
        plt.tight_layout()
        plt.savefig(OUT / "cm_v3.png", dpi=150)
        plt.show()
        print("→ FP=22 (real bị nhận nhầm fake), FN=28 (fake bị nhận nhầm real).")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [15]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # --- LEGACY SECTION (external RunPod benchmark) -------------------------
    # These cells were authored against /workspace/... predictions that are
    # not part of this repository. They are preserved verbatim below but are
    # skipped when their inputs are absent. Nothing is fabricated.
    _legacy_needed = [n for n in ('labels', 'probs', 'preds') if n not in dir()]
    if _legacy_needed:
        print('LEGACY SECTION SKIPPED - external benchmark predictions absent:',
              _legacy_needed)
        print('  See the SESSION-2 addendum below for the canonical, reproducible\n'
              '  evaluation on the current protocol.')
    else:
        # ---- ROC + Precision-Recall ----
        fpr, tpr, _ = roc_curve(labels, probs)
        pr_prec, pr_rec, _ = precision_recall_curve(labels, probs)

        fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

        ax = axes[0]
        ax.plot(fpr, tpr, lw=2.2, color="#c33", label=f"finetune v3 (AUC={auc*100:.2f}%)")
        ax.plot([0, 1], [0, 1], "k--", alpha=.4)
        ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
        ax.set_title("ROC Curve"); ax.legend(loc="lower right"); ax.grid(alpha=.3)

        ax = axes[1]
        ax.plot(pr_rec, pr_prec, lw=2.2, color="#28a", label="PR curve")
        ax.axhline(prec, color="r", ls="--", alpha=.5, label=f"precision {prec*100:.1f}%")
        ax.set_xlabel("Recall (fake)"); ax.set_ylabel("Precision")
        ax.set_title("Precision-Recall Curve"); ax.legend(loc="upper right"); ax.grid(alpha=.3)

        plt.tight_layout()
        plt.savefig(OUT / "roc_prc_v3.png", dpi=150)
        plt.show()
        print("→ AUC 99.70%: tách real/fake gần như hoàn hảo; PRC cho thấy độ tin cậy trên nhãn fake.")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [16]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # --- LEGACY SECTION (external RunPod benchmark) -------------------------
    # These cells were authored against /workspace/... predictions that are
    # not part of this repository. They are preserved verbatim below but are
    # skipped when their inputs are absent. Nothing is fabricated.
    _legacy_needed = [n for n in ('labels', 'probs', 'preds') if n not in dir()]
    if _legacy_needed:
        print('LEGACY SECTION SKIPPED - external benchmark predictions absent:',
              _legacy_needed)
        print('  See the SESSION-2 addendum below for the canonical, reproducible\n'
              '  evaluation on the current protocol.')
    else:
        # ---- per-method accuracy: v3 vs baseline (từ CSV đã lưu) + cảnh báo leak ----
        def read_pm(p):
            with open(p) as f:
                return {r["Method"]: float(r["Accuracy"]) for r in csv.DictReader(f)}

        pm_base = read_pm(PM_BASE)
        pm_v2   = read_pm(PM_V2)
        pm_v3   = read_pm(PM_V3)

        # các method quan tâm (chỗ sụp baseline + real)
        focus = ["faceswap", "starganv2", "whichfaceisreal", "facedancer", "sadtalker",
                 "fsgan", "wav2lip", "e4s", "simswap", "blendface", "lia", "inswap", "real"]
        focus = [m for m in focus if m in pm_base and m in pm_v3]

        # số ảnh test của từng method bị leak (từ 1.3)
        leak_n = Counter(test[i]["method"] for i in leaked_idx)

        print(f"\n{'method':<20}{'baseline':>9}{'v2':>8}{'v3':>8}{'leak(test)':>11}  ghi chú")
        for m in focus:
            lk = leak_n.get(m, 0)
            n_te = te_method.get(m, 0)
            note = ""
            if m in ("whichfaceisreal", "CollabDiff") and lk == n_te and n_te > 0:
                note = "⚠️ 100% test trong train → KHÔNG đo được"
            elif lk > 0:
                note = f"⚠️ {lk}/{n_te} test trong train"
            print(f"{m:<20}{pm_base[m]:>9.1f}{pm_v2[m]:>8.1f}{pm_v3[m]:>8.1f}{lk:>5}/{n_te:<5}  {note}")

        x = np.arange(len(focus)); w = 0.38
        fig, ax = plt.subplots(figsize=(12.5, 5.2))
        ax.bar(x - w/2, [pm_base[m] for m in focus], w, label="baseline v5", color="#c8967a")
        ax.bar(x + w/2, [pm_v3[m] for m in focus], w, label="finetune v3", color="#4a7fb5")
        leak_tick = ["⚠️ " + m if leak_n.get(m, 0) > 0 else m for m in focus]
        ax.set_xticks(x); ax.set_xticklabels(leak_tick, rotation=38, ha="right")
        ax.set_ylabel("Accuracy %"); ax.set_ylim(50, 102)
        ax.axhline(100, color="k", ls="--", lw=.8, alpha=.4)
        ax.set_title("Per-method accuracy: baseline v5 vs finetune v3 (⚠️ = method có ảnh test lọt vào train)")
        ax.legend(); ax.grid(alpha=.3, axis="y")
        plt.tight_layout()
        plt.savefig(OUT / "per_method_v3_vs_base.png", dpi=150)
        plt.show()
        print("""
        → ĐỌC KẾT QUẢ TRUNG THỰC:
          • faceswap 62.96→88.89  : CLEAN (0 ảnh leak) — cải thiện THẬT, là mục tiêu chính của v3. ✅
          • whichfaceisreal, CollabDiff : 100% ảnh test trong train → điểm v3 là memorize, BỎ QUA.
          • starganv2: 33/40 ảnh test trong train → điểm 95% không đáng tin (chỉ 7 ảnh sạch).
          • facedancer 74→92.6, fsgan/simswap/blendface/lia/wav2lip/e4s/inswap: không leak → cải thiện THẬT.""")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


In [17]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    # --- LEGACY SECTION (external RunPod benchmark) -------------------------
    # These cells were authored against /workspace/... predictions that are
    # not part of this repository. They are preserved verbatim below but are
    # skipped when their inputs are absent. Nothing is fabricated.
    _legacy_needed = [n for n in ('labels', 'probs', 'preds') if n not in dir()]
    if _legacy_needed:
        print('LEGACY SECTION SKIPPED - external benchmark predictions absent:',
              _legacy_needed)
        print('  See the SESSION-2 addendum below for the canonical, reproducible\n'
              '  evaluation on the current protocol.')
    else:
        # ---- chi tiết faceswap: từng ảnh test, prob v3 ----
        fs_idx = [i for i, x in enumerate(test) if x["method"] == "faceswap"]
        print(f"faceswap test: {len(fs_idx)} ảnh")
        print(f"\n{'identity':<28}{'nguồn':<10}{'prob(fake)':>10}{'pred':>6}")
        rows_out = []
        for i in fs_idx:
            ident = test[i]["identity"]
            src = "FF++" if ident.startswith("ffc:") else ("VoxCeleb" if ident.startswith("oth:") else "Celeb-DF")
            prob = float(probs[i]); pred = "FAKE" if prob >= 0.5 else "REAL"
            mark = "✅" if pred == "FAKE" else "❌"
            rows_out.append((ident, src, prob, pred, mark))
            print(f"{ident:<28}{src:<10}{prob:>10.3f}{pred:>6} {mark}")

        fs_correct = sum(1 for r in rows_out if r[3] == "FAKE")
        fs_leak = [i for i in fs_idx if i in leaked_idx]
        print(f"\nfaceswap accuracy (v3): {fs_correct/len(fs_idx)*100:.2f}% ({fs_correct}/{len(fs_idx)})")
        print(f"faceswap ảnh test bị leak: {len(fs_leak)}  → {'CLEAN ✅ (không ảnh nào trong train)' if not fs_leak else 'có leak!'}")
        print("""
        → So với baseline/v2 (10 miss): 7/10 đã sửa. 3 ca còn lại:
          • ffc:701  — FF++ frame cực sắc (sharpness 527), model vẫn tự tin REAL.
          • oth:id3_id23 — VoxCeleb ngoài phân phối train.
          • cdc:id10 — Celeb-DF sát ngưỡng (prob ~0.48).
        → Vì faceswap 0 leak nên cải thiện 62.96→88.89 là kết quả tổng quát THẬT.""") 


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


---

## 3.2 Tổng kết

### Bảng chính (metric trên test)

| Chỉ số | baseline v5 | **v3 (with-leak 2,354)** | **v3 CLEAN (loại 95 ảnh leak)** |
|---|---|---|---|
| Test accuracy | 95.37% | 97.88% | **97.96%** |
| ROC-AUC | 99.35% | 99.70% | ~99.7% |
| Precision | 97.26% | 98.12% | **97.96%** |
| Recall (fake) | 93.37% | 97.62% | **97.78%** |
| FP / FN | 31 / 78 | 22 / 28 | 22 / 24 |
| faceswap | 62.96% | **88.89%** | **88.89%** (0 leak) |

### Kết luận trung thực

1. **faceswap — cải thiện THẬT và sạch:** 62.96% → 88.89% (7/10 miss sửa). Không ảnh faceswap
   nào nằm trong train → đây là kết quả tổng quát, đúng mục tiêu v3. Nhờ **+8,000 data
   identity-disjoint + sampler faceswap-focused (P=0.35)**.
2. **95 ảnh test bị LEAK vào train v2/v3** (phát hiện bằng MD5 toàn bộ): starganv2, whichfaceisreal,
   CollabDiff — từ bổ sung `df-40-test-full`. **whichfaceisreal & CollabDiff 100% test trong train
   → điểm per-method của 2 method này là memorize, KHÔNG đo được.** starganv2 chỉ còn 7 ảnh sạch.
3. **Tổng thể vẫn tốt:** loại 95 ảnh leak, accuracy **97.96%** (≈ không đổi) → kết luận v3 tốt hơn
   baseline vẫn đứng vững; chỉ cần **bỏ qua** các con số per-method của 3 method kể trên.
4. Các method khác (facedancer, fsgan, simswap, blendface, wav2lip, e4s, inswap, …) **không leak**
   → cải thiện là thật.

> ⚠️ **Bài học:** benchmark v5 gốc được chứng nhận "0% leak" so với train v5, nhưng **mỗi lần thêm
> data phải chạy lại MD5 trên toàn bộ train mới**. Lần này v2 build bổ sung `test_full` mà bộ trích
> identity token không nhận format `starganv2_clean_N` / `efs:...:N` → 95 frame trùng byte lọt vào.
> Đã ghi nhận để lần sau dùng `re.findall(r"\d+", identity)` cho **mọi** format (như `expand_faceswap_v3.py`).

### Files kết quả
- Checkpoint v3: `hoangtuan/deepfake-ViT/experiments/checkpoints/exp05_v5_weakfix_v3/best_model.pt`
- Báo cáo: `v5_weakfix_v3_training_report.json`, `v5_weakfix_v3_per_method_accuracy.csv`
- Hình từ notebook này: `quangmanh/deepfake/experiments/results/courseWorkCheck/*.png`
- Dữ liệu leak: `md5_leak.json`, `leaked_idx.json` (95 index test bị trùng byte)

### Tái lập
```bash
REPO=/workspace/hoangtuan/deepfake-ViT
PY=$REPO/.venv/bin/python
$PY $REPO/scripts/expand_faceswap_v3.py
$PY $REPO/scripts/finetune_v5_weakfix_v3.py --init-ckpt $REPO/experiments/checkpoints/exp05_v5_weakfix/best_model.pt
$PY $REPO/scripts/eval_v5_weakfix_v3_report.py
```

---

*Notebook tự sinh theo `notebooks/guide.md`; mọi số liệu tính trực tiếp từ model + dữ liệu, không ghi tay.
Phát hiện leak 95 ảnh bằng MD5 toàn bộ (mục 1.3) là điểm mới so với báo cáo cũ (04/05).*

## Final Protocol Check Against Session 2 EDA Reference

This coursework-facing section reconciles the notebook with the provided Session 2 EDA reference and the latest GitHub-aware project state. It avoids contradictions by showing exactly which local/remote artifacts are available before claiming Session 2 counts or evaluation results.

In [18]:
# --- LEGACY SECTION (v5_weakfix corpus / external benchmark) -----------
# Preserved verbatim. Runs only when the legacy inputs are present.
if not LEGACY_DATA_AVAILABLE:
    print('LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.')
else:
    from pathlib import Path
    import json
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    try:
        from IPython.display import display
    except Exception:
        def display(x): print(x.to_string() if hasattr(x, "to_string") else x)

    PROJECT_ROOT = Path(os.getcwd()).resolve()
    while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    if not (PROJECT_ROOT / "data").exists():
        PROJECT_ROOT = Path("/Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT")

    WEAK8 = {"faceswap", "deepfake_faceswap", "wav2lip", "sadtalker", "fsgan", "facedancer", "inswap", "mobileswap"}
    EXPECTED = pd.DataFrame([
        ["Session 2 train", 123582, 29557, 94025, "42 fake methods / 9 real sources"],
        ["Session 2 val", 6302, None, None, "same train-like imbalance"],
        ["Session 2 full test", 50084, 25042, 25042, "balanced 1:1"],
        ["Session 2 balanced test", 21446, 10723, 10723, "balanced eval protocol"],
    ], columns=["reference_population", "rows", "real", "fake", "protocol_note"])
    print(f"Project root: {PROJECT_ROOT}")
    print("Session 2 reference targets from the provided PDF:")
    display(EXPECTED)

    cfg_path = PROJECT_ROOT / "data/protocol/protocol_config.json"
    meta_path = PROJECT_ROOT / "data/protocol/protocol_metadata.json"
    meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
    print("Current local canonical protocol:")
    display(pd.json_normalize(meta, sep=".").T.rename(columns={0:"value"}).head(20) if meta else pd.DataFrame({"status":["metadata missing"]}))

    rows = []
    for split in ["train", "val", "test"]:
        p = PROJECT_ROOT / f"data/protocol/{split}_detailed.csv"
        if p.exists():
            df = pd.read_csv(p)
            rows.append({"population": f"local canonical {split}", "rows": len(df), "real": int((df.label == 0).sum()), "fake": int((df.label == 1).sum()), "fake_methods": df[df.label == 1].method.nunique(), "identities": df.identity.nunique() if "identity" in df else np.nan})
    for name, rel in [("local full manifest", "test_data_v3/manifest.csv"), ("local legacy balanced test", "data/splits/test_balanced.csv"), ("Session 2 balanced CSV", "data/deepfake_test_suite_full_50k/splits/test_coursework_44methods_balanced_zero_leakage.csv")]:
        p = PROJECT_ROOT / rel
        if p.exists():
            df = pd.read_csv(p)
            rows.append({"population": name, "rows": len(df), "real": int((df.label == 0).sum()), "fake": int((df.label == 1).sum()), "fake_methods": df[df.label == 1].method.nunique() if "method" in df else np.nan, "identities": df.identity.nunique() if "identity" in df else np.nan})
    actual = pd.DataFrame(rows)
    display(actual)

    print("Reproducibility status: the notebook uses the current canonical protocol if Session 2 CSVs are absent; when the remote Session 2 artifacts are checked out, the same cells detect and summarize them without creating a new split.")

    # Coursework-facing evidence chain.
    chain = pd.DataFrame([
        ["Data", "Show exact split counts, real/fake balance, methods and real sources from CSVs", "Prevents reporting PDF numbers when the local protocol is different"],
        ["Data quality/leakage", "Identity intersections, duplicate paths, and stored duplicate reports", "Prevents leakage-inflated validation claims"],
        ["Distribution/long tail", "Per-method pools, percentages, bottom quartile, weak8 marker", "Explains why weak methods need targeted exposure"],
        ["Training implication", "Exposure accounting for A0/A1 sampler policies", "Connects imbalance to sampler design without implementing another sampler"],
        ["Evaluation evidence", "Per-method/source errors, paired McNemar when predictions are available", "Separates association from causality and verifies improvements on the same samples"],
    ])
    chain.columns = ["stage", "notebook evidence", "why it matters"]
    display(chain)

    course = PROJECT_ROOT / "experiments/results/coursework_vs"
    if course.exists():
        summaries = []
        for p in sorted(course.glob("eval_*.json")):
            obj = json.loads(p.read_text())
            summaries.append({"model": p.stem.replace("eval_", ""), "accuracy": obj.get("accuracy"), "precision": obj.get("precision"), "recall": obj.get("recall"), "f1": obj.get("f1"), "auc": obj.get("roc_auc"), "real_acc": obj.get("real_acc"), "FP": obj.get("FP"), "FN": obj.get("FN")})
        if summaries:
            display(pd.DataFrame(summaries).sort_values("accuracy", ascending=False))
    else:
        print("coursework_vs result folder is not present in this working tree yet; it exists on origin/main and will be picked up after sync.")


LEGACY SECTION SKIPPED - required legacy inputs are not in this repo.


---
# SESSION-2 EDA ADDENDUM - Cross-Notebook Consistency Verification

Verifies the coursework-facing notebook agrees with the canonical protocol and
with every other notebook. Verification only: nothing is modified here.

### Provenance convention used in this addendum

Every figure below is tagged:

* **MEASURED** - computed from an artifact present in this repository.
* **REFERENCE** - quoted from `session2_finetune_data_eda.pdf` because the
  underlying raw data is *not* in this repository. Never presented as measured.

The Session-2 **training** corpus (`finetune_plus_train.csv`, ~123.6k images)
and the 50k test suite are not committed here. The Session-2 **evaluation**
artifacts are, under `experiments/results/coursework_vs/`, so all
evaluation-side findings are reproduced from real paired predictions.

In [19]:
# --- Session-2 addendum bootstrap (shared implementation, no duplication) ---
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from src.data import session2_eda as S2
from src.data import protocol as P
from src.data import loader

pd.set_option("display.width", 200)

# ---- Shared consistency banner: identical in all five notebooks. ----
# Values come from src.data.session2_eda (REFERENCE + measured .npz preds).
# If any notebook ever printed different values here, the notebooks would be
# describing different data. They cannot, because they all read one module.
print(S2.consistency_banner())
print()
print(P.describe())
print()
print(loader.describe())
print()
print("KNOWN LIMITATIONS OF THE ACTIVE PROTOCOL")
for _lim in P.KNOWN_LIMITATIONS:
    print(f"  - {_lim}")
print()

# Be explicit about which reference findings can and cannot be measured here.
print(S2.describe())

Session 2 Analysis Consistency
Session-2 dataset identity : finetune_plus (Session-2 finetune corpus / session2_finetune_data_eda.pdf)
  total images (train)   : 123,582  (29,557 real / 94,025 fake)  [REFERENCE]
  fake methods           : 42  | real sources: 9  [REFERENCE]
  canonical test composition [REFERENCE]:
    val=6,302  test_full=50,084  test_balanced=21,446 (1:1)
  balanced test MEASURED : n=21,446  real=10,723  fake=10,723  (1:1.00)
  analysis/prediction artifacts : experiments/results/coursework_vs/*.npz  [AVAILABLE]

Protocol distinction (never mix corpora):
  - Session-2 analysis uses the Session-2 balanced test / .npz preds above.
  - Local active protocol is identity_clean_v1 (separate corpus; ~4.5k imbalanced test).

CANONICAL DATASET PROTOCOL
  name              : identity_clean_v1
  strategy          : identity-disjoint
  seed              : 42
  dataset root      : /Users/pickapu/Documents/PyCharmMiscProject/deepfake-ViT/test_data_v3
  protocol dir      : /Users/pic

## E1. Canonical protocol and label semantics

In [20]:
proto = P.load_protocol(); meta = P.protocol_metadata()
checks = {
    "protocol is identity_clean_v1": proto.name == "identity_clean_v1",
    "strategy identity-disjoint": meta["split_strategy"] == "identity-disjoint",
    "seed == 42": meta["random_seed"] == 42,
    "labels 0=real, 1=fake": P.LABEL_MAPPING == {0: "real", 1: "fake"},
    "train == 20,991": meta["train_size"] == 20991,
    "val   == 4,498": meta["val_size"] == 4498,
    "test  == 4,499": meta["test_size"] == 4499,
    "sizes match CSVs": loader.split_sizes() == {"train": 20991, "val": 4498, "test": 4499},
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "canonical protocol drift detected"

  [PASS] protocol is identity_clean_v1
  [PASS] strategy identity-disjoint
  [PASS] seed == 42
  [PASS] labels 0=real, 1=fake
  [PASS] train == 20,991
  [PASS] val   == 4,498
  [PASS] test  == 4,499
  [PASS] sizes match CSVs


## E2. Dataset arithmetic and leakage

In [21]:
manifest = loader.load_manifest()
total = sum(loader.split_sizes().values())
removed = len(manifest) - total
print(f"manifest {len(manifest):,} - removed {removed:,} = protocol {total:,}  "
      f"-> {len(manifest)-removed == total}")

log = _ROOT / "experiments/results/eda_real_data/removed_exact_duplicates.csv"
if log.exists():
    n = len(pd.read_csv(log))
    print(f"audit log rows {n:,} == removed {removed:,} -> {n == removed}")
    print("Every removed duplicate is logged; nothing silently deleted.")

det = {s: loader.load_split(s, detailed=True) for s in P.SPLITS}
for key in ["identity", "video"]:
    sets = {s: set(d[key]) for s, d in det.items()}
    ov = {f"{a}-{b}": len(sets[a] & sets[b])
          for a, b in [("train", "val"), ("train", "test"), ("val", "test")]}
    tag = "PASS (must be 0)" if key == "identity" and sum(ov.values()) == 0 else (
          "KNOWN LIMITATION (not safe)" if key == "video" else "FAIL")
    print(f"{key:9s} overlap {ov}  -> {tag}")

manifest 30,691 - removed 703 = protocol 29,988  -> True
audit log rows 703 == removed 703 -> True
Every removed duplicate is logged; nothing silently deleted.


identity  overlap {'train-val': 0, 'train-test': 0, 'val-test': 0}  -> PASS (must be 0)


video     overlap {'train-val': 1509, 'train-test': 1542, 'val-test': 930}  -> KNOWN LIMITATION (not safe)


## E3. Evaluation populations are the right ones

In [22]:
PRED = _ROOT / "experiments/results/baseline/final/evaluation/test_predictions.csv"
if PRED.exists():
    preds = pd.read_csv(PRED)
    try:
        loader.verify_matches_protocol(preds, "test")
        print(f"PASS: local baseline predictions ({len(preds):,}) == canonical test split")
    except ValueError as e:
        print("FAIL:", e)
else:
    print("local baseline predictions not found")

if S2.availability().eval_preds:
    c = S2.test_composition()
    print(f"PASS: Session-2 balanced test measured at n={c['n_total']:,} "
          f"({c['n_real']:,} real / {c['n_fake']:,} fake, 1:{c['real_fake_ratio']:.2f})")
    print("      matches the reference figure of 21,446 at 1:1")
print()
print("NOTE: two distinct corpora are in play. They are never merged:")
print("  - local identity_clean_v1 : 4,499-image test, ~96% fake -> use MCC")
print("  - Session-2 balanced      : 21,446-image test, 1:1     -> accuracy is fair")

PASS: local baseline predictions (4,499) == canonical test split

PASS: Session-2 balanced test measured at n=21,446 (10,723 real / 10,723 fake, 1:1.00)
      matches the reference figure of 21,446 at 1:1

NOTE: two distinct corpora are in play. They are never merged:
  - local identity_clean_v1 : 4,499-image test, ~96% fake -> use MCC
  - Session-2 balanced      : 21,446-image test, 1:1     -> accuracy is fair


## E4. Headline results, each on its own corpus

In [23]:
import json as _json
mp = _ROOT / "experiments/results/baseline/final/evaluation/metrics.json"
if mp.exists():
    m = _json.load(open(mp))
    print("LOCAL identity_clean_v1 (imbalanced test - accuracy is misleading)")
    print(f"   accuracy {m['accuracy']:.4f} | balanced acc {m['balanced_accuracy']:.4f} "
          f"| MCC {m['mcc']:.4f} | real recall {m['real_recall']:.4f}")
    print(f"   TN/FP/FN/TP = {m['tn']}/{m['fp']}/{m['fn']}/{m['tp']}")

if S2.availability().eval_preds:
    print("\nSESSION-2 balanced test (1:1 - accuracy is fair)")
    display(S2.metrics_table()[["model", "acc%", "f1", "AUC", "real_acc",
                                "fake_recall", "FP", "FN"]].round(4))


SESSION-2 balanced test (1:1 - accuracy is fair)


,model,acc%,f1,AUC,real_acc,fake_recall,FP,FN
0,ViT-Plus A0 (old sampler),97.9110,0.9791,0.9979,0.9805,0.9777,209,239
1,ViT-Plus A1 (weak_family),98.4706,0.9848,0.9986,0.9792,0.9902,223,105
2,ConvNeXt (finetuned ref),99.2166,0.9921,0.9998,0.9979,0.9864,22,146
3,ViT-S/16 (session-1 ref),97.2349,0.9724,0.9960,0.9707,0.9740,314,279
4,Pretr ViT-Plus (frozen probe),89.4666,0.8955,0.9624,0.8869,0.9025,1213,1046
5,Pretr ConvNeXt (frozen probe),87.7739,0.8748,0.9553,0.9011,0.8544,1061,1561


## E5. Reference findings: reproduced, represented, or unavailable

In [24]:
av = S2.availability()
rows = [
    ("1. Train composition 123,582 / 94,025 fake / 29,557 real", "REFERENCE only",
     "raw Session-2 train CSV absent locally"),
    ("2. 9 real sources, diversity beyond FF++/Celeb-DF", "REFERENCE only",
     "shown in nb00 A2, clearly labelled"),
    ("3. 42 methods, long-tail pools", "REFERENCE only",
     "shown in nb00 A3; weak family PARSED from sampler"),
    ("4. Dimensions 256 / 512 / 178x218", "PARTIAL",
     "local dims MEASURED in nb00 A4; Session-2 dims REFERENCE"),
    ("5. Identity ~2.5-3.4 img/id, train-val identity overlap 0", "PARTIAL",
     "Session-2 counts REFERENCE; local identity+leakage MEASURED"),
    ("6. Splits: train/val/test-full/test-balanced", "PARTIAL",
     "balanced test 21,446 @1:1 MEASURED from predictions"),
    ("7. Sampler exposure A0 vs A1 (dfs 0.056 -> 0.52, x9)", "REPRODUCED",
     "recomputed with the project sampler rules, matches reference"),
    ("8. A1 vs A0 McNemar p=3.6e-10, fake p=3e-21, real p=0.31", "REPRODUCED",
     "computed from local paired predictions"),
    ("9. Per-method gains concentrated on the 8 weak methods", "REPRODUCED",
     "nb02_error_analysis C2/C3"),
    ("10. Pretrained probe ~89.5/87.8% vs finetuned ~98.5/99.2%", "REPRODUCED",
     "nb02_error_analysis C6"),
    ("11. Probes weakest on Face Swap / weak family", "REPRODUCED",
     "nb02_error_analysis C6"),
    ("12. Real-source difficulty (ff++_real hardest)", "REPRODUCED",
     "nb02_error_analysis C4"),
]
cov = pd.DataFrame(rows, columns=["reference finding", "status", "where / why"])
display(cov)
print(cov["status"].value_counts().to_string())
print("\nNo reference finding is presented as measured when it is not.")

,reference finding,status,where / why
0,"1. Train composition 123,582 / 94,025 fake / 2...",REFERENCE only,raw Session-2 train CSV absent locally
1,"2. 9 real sources, diversity beyond FF++/Celeb-DF",REFERENCE only,"shown in nb00 A2, clearly labelled"
2,"3. 42 methods, long-tail pools",REFERENCE only,shown in nb00 A3; weak family PARSED from sampler
3,4. Dimensions 256 / 512 / 178x218,PARTIAL,local dims MEASURED in nb00 A4; Session-2 dims...
4,"5. Identity ~2.5-3.4 img/id, train-val identit...",PARTIAL,Session-2 counts REFERENCE; local identity+lea...
5,6. Splits: train/val/test-full/test-balanced,PARTIAL,"balanced test 21,446 @1:1 MEASURED from predic..."
6,7. Sampler exposure A0 vs A1 (dfs 0.056 -> 0.5...,REPRODUCED,"recomputed with the project sampler rules, mat..."
7,"8. A1 vs A0 McNemar p=3.6e-10, fake p=3e-21, r...",REPRODUCED,computed from local paired predictions
8,9. Per-method gains concentrated on the 8 weak...,REPRODUCED,nb02_error_analysis C2/C3
9,10. Pretrained probe ~89.5/87.8% vs finetuned ...,REPRODUCED,nb02_error_analysis C6


status
REPRODUCED        6
REFERENCE only    3
PARTIAL           3

No reference finding is presented as measured when it is not.


## E6. Consistency verdict

* One canonical protocol (`identity_clean_v1`), one label mapping
  (`0=real, 1=fake`), one preprocessing (256x256) across all five notebooks.
* Split sizes reconcile with the manifest and the duplicate audit log.
* Identity and exact-duplicate cross-split overlap are 0.
* Video overlap and un-removed near-duplicates remain **documented limitations**
  and are never described as safe.
* Val/test are never rebalanced; balancing is training-side only.
* The two corpora are reported separately and never mixed.

**Known limitations carried forward**

1. Video/source overlap in `identity_clean_v1`.
2. 4,215 cross-split near-duplicate groups, quantified but not removed.
3. Session-2 raw training corpus absent locally - train-side composition is
   reference-only.
4. Local baseline is a 1-epoch-derived 8-epoch run whose MCC was still rising:
   not converged.